# Appendix C — The GMS substrate & calibration

This appendix makes the book stand on its own. It recaps the six store
primitives, shows how a trained store is assembled from triples, and gives
the **threshold-calibration method**: cohort -> sweep -> operating point
under a false-allow ceiling. Geometry internals (rotors, admissibility
caps, holonomy proofs) are deferred to the GMS monograph.

The calibration sweep at the end runs **CPU-only** with a deterministic
injected encoder, so it reproduces a chosen operating point without a GPU,
without the trained store, and without Qwen.

In [ ]:
import os, sys
KNOWLYTIX_SRC = os.environ.get("KNOWLYTIX_SRC", "/path/to/GMS-knowlytix")
sys.path.insert(0, KNOWLYTIX_SRC)

## C.1 The six store primitives

Everything the RAG pipeline does to the graph routes through six methods on
`GMSExpertStore`. You do not need the geometry to use them; you need to know
what each returns and when to reach for it.

| Primitive | Question it answers | Return |
|---|---|---|
| `score_triple(h, r, t)` | How well does this asserted fact fit the trained geometry? | geodesic score (float) or `None` |
| `lookup_enm(cat, id)` | What is the **exact** stored number? | byte-exact value (float) or `None` |
| `query_triples(head, relation, tail)` | Which asserted edges match this pattern? | list of triples |
| `link_predict(head, relation, ...)` | What tail does the geometry *predict* (no asserted edge)? | ranked candidates |
| `check_holonomy(path, ...)` | Is a relation composition path consistent? | residual / verdict |
| `tension_energy(a, b)` | How contradictory are two entities? | energy (float) or `None` |

The design rule, repeated throughout the book: **`lookup_enm` for numbers,
asserted `query_triples` before `link_predict` for facts, `tension_energy`
for contradiction.** A predicted link is a hypothesis, not a fact (see
Chapter 8).

In [ ]:
from knowlytix.knowledge.store import GMSExpertStore

# The public surface this appendix recaps. We assert the methods exist;
# we do NOT load a trained store here (one GPU is shared across authors).
PRIMITIVES = (
    "score_triple", "lookup_enm", "query_triples",
    "link_predict", "check_holonomy", "tension_energy",
)
for name in PRIMITIVES:
    assert callable(getattr(GMSExpertStore, name)), name
print("six primitives present:", ", ".join(PRIMITIVES))

Expected output:
```
six primitives present: score_triple, lookup_enm, query_triples, link_predict, check_holonomy, tension_energy
```

## C.2 Building a store

Two entry points assemble a trained `GMSExpertStore`, both in
`knowlytix.knowledge.geode.rag`:

- `build_rag_store(md_path, config, ...)` — the full ingest path: parse the
  document, extract triples, populate ENM, run the GEODE self-correction
  loop, then train. This is what `scripts/build_store.py` calls to produce
  `data/gms_annual_report_store/`.
- `store_from_triples(md_path, triples, config, ...)` — skip extraction and
  train directly from a triple list you already trust.

Both are GPU/training operations. We do **not** call them here; the cell
below is marked for the lead to execute in CI. The grounding numbers it
prints are the canonical FY2025 figures (segment total revenue 355.0).

In [ ]:
# CI-ONLY (GPU): build the trained store from the annual report.
# Do not run during authoring; the lead executes this in CI.
RUN_GPU = os.environ.get("GMS_RUN_GPU") == "1"
if RUN_GPU:
    from knowlytix.knowledge.geode.rag import build_rag_store
    from knowlytix.knowledge.geode.rag import DocGMSConfig  # config dataclass
    store = build_rag_store(os.path.join(
        os.path.join(os.path.dirname(os.getcwd()), "code") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd(),
        "data", "annual_report.md"), DocGMSConfig())
    # Numbers come from data/corpus_facts.md (byte-exact ENM).
    assert store.lookup_enm("segment_performance", "Total/All/Revenue") == 355.0
    assert store.lookup_enm("income_statement", "Net Income/FY2025") == 70.0
    print("built store: total revenue =",
          store.lookup_enm("segment_performance", "Total/All/Revenue"))
else:
    print("skipped GPU build (set GMS_RUN_GPU=1 in CI)")

## C.3 The threshold-calibration method

Embedding binding (Chapter 7) resolves a paraphrase like *topline* to the
canonical relation `has_revenue` by cosine similarity. That comparison needs
a **threshold**: bind above it, refuse below it. Setting the threshold by
eye is the anti-pattern. The method is:

1. **Cohort.** Collect labeled `(term, expected_entity)` *positives* that
   should bind and *negatives* that should bind to nothing.
2. **Sweep.** Walk a grid of candidate thresholds.
3. **Operating point.** Pick the threshold that best separates the two
   classes. Tightening the false-allow ceiling (refusing more negatives)
   pushes the threshold up.

`calibrate_bind_threshold(binder, positives, negatives, grid=...)` does the
sweep and returns `(best_threshold, accuracy)`, also setting it on the
binder. This is the same operating-point recipe used for the accept/abstain
cuts in Chapter 12 and for the agent book's gates (cross-ref App C there).

The cell below is **fully runnable on CPU**: it injects a tiny deterministic
encoder and a stub adapter, so no GPU, no trained store, and no Qwen are
touched. The vocabulary mirrors the Northwind store's relations.

In [ ]:
import numpy as np
from knowlytix.knowledge.rag.binding import TripleBinder
from knowlytix.knowledge.rag.eval import calibrate_bind_threshold

# A deterministic toy encoder: each text maps to a fixed unit vector.
# Synonyms sit near their canonical relation; unrelated terms sit far away.
_VEC = {
    # canonical graph relations (mirror data/corpus_facts.md)
    "has_revenue":   [1.0, 0.0, 0.0],
    "has_headcount": [0.0, 1.0, 0.0],
    "has_region":    [0.0, 0.0, 1.0],
    # positive synonyms -> should bind to has_revenue / has_headcount
    "topline":       [0.97, 0.10, 0.0],
    "sales":         [0.95, 0.05, 0.0],
    "staff count":   [0.05, 0.98, 0.0],
    # negatives -> should bind to nothing (placed deliberately close to
    # has_revenue so the sweep has to RAISE the threshold to refuse them)
    "weather":       [0.60, 0.55, 0.0],
    "breakfast":     [0.62, 0.50, 0.0],
}

def toy_encoder(texts):
    out = []
    for t in texts:
        v = np.asarray(_VEC.get(t.strip().lower(), [0.33, 0.33, 0.33]),
                       dtype=np.float32)
        out.append(v / (np.linalg.norm(v) + 1e-9))
    return np.vstack(out)

class _StubAdapter:  # the binder only needs the relation/entity vocab
    relation_to_idx = {"has_revenue": 0, "has_headcount": 1, "has_region": 2}
    entity_to_idx = {"has_revenue": 0, "has_headcount": 1, "has_region": 2}

class _StubStore:
    adapter = _StubAdapter()
    def fuzzy_match_entity(self, name):
        # force the embedding path: no fuzzy hit for paraphrases
        return name if name in _StubAdapter.entity_to_idx else None

binder = TripleBinder(_StubStore(), mode="embedding", encoder=toy_encoder,
                      bind_threshold=0.5, bind_margin=0.05)
print("binder mode:", binder.mode, "| start threshold:", binder.bind_threshold)

In [ ]:
# The labeled cohort: terms that SHOULD bind, and terms that should NOT.
positives = [
    ("topline", "has_revenue"),
    ("sales", "has_revenue"),
    ("staff count", "has_headcount"),
]
negatives = ["weather", "breakfast"]

best_th, acc = calibrate_bind_threshold(binder, positives, negatives)
print(f"chosen operating point: threshold={best_th:.2f}  accuracy={acc:.2f}")
print("binder threshold now set to:", binder.bind_threshold)

Expected output (deterministic on CPU):
```
chosen operating point: threshold=0.80  accuracy=1.00
binder threshold now set to: 0.8
```
The negatives sit at cosine ~0.74-0.78 from `has_revenue`, so a lax
threshold (0.50) would *false-allow* them. The sweep raises the threshold to
0.80 — above the negatives, below the synonyms (~0.99) — which binds every
synonym and refuses every negative (accuracy 1.00). On the real Northwind
store the encoder is MiniLM and the cohort comes from
`data/eval_cohort.json`, but the method is identical.

## C.4 Honest limits

Calibration tunes a decision boundary; it does not create separability. If
positives and negatives overlap in embedding space, no threshold recovers
100% accuracy — the sweep returns the *least-bad* point, not a correct one.
A calibrated threshold is only as honest as its cohort: calibrate on hand-
labeled pairs, never on the system's own accepted outputs (that bakes in
today's mistakes). And calibration says nothing about whether a *fact* is
true — that is the verifier's job (Chapter 10), not the binder's. Finally,
this appendix recaps the substrate's *interface*; the geometric guarantees
behind `score_triple` and `check_holonomy` live in the GMS monograph.

## C.5 Self-check

Re-running the calibration must reproduce the same operating point and that
point must perfectly separate the labeled cohort.

In [ ]:
# Reproduce the operating point and prove it separates the cohort.
binder.bind_threshold = best_th
from knowlytix.knowledge.rag.query_triples import QueryTriple

pos_ok = all(binder.bind(QueryTriple(t, "_", "?")).head == exp
             for t, exp in positives)
neg_ok = all(binder.bind(QueryTriple(t, "_", "?")).head is None
             for t in negatives)

assert acc == 1.0, f"cohort not separable at acc={acc}"
assert pos_ok, "a positive synonym failed to bind at the chosen threshold"
assert neg_ok, "a negative term bound when it should have been refused"
print("OK: operating point", round(best_th, 2),
      "separates the labeled cohort (acc=1.00)")